In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [7]:
from src.data_loader import load_raw_data

df = load_raw_data()

In [8]:
from src.preprocessing import clean_data

df = clean_data(df)
print(df.shape)

(1337, 7)


In [13]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["charges"])
y = df["charges"]

Feature Engineering

In [ ]:
X = X.copy()

X["bmi_obese"] = (X["bmi"] >= 30).astype(int)
X[["bmi", "bmi_obese"]].head(10)

,bmi,bmi_obese
0,27.900,0
1,33.770,1
2,33.000,1
3,22.705,0
4,28.880,0
5,25.740,0
6,33.440,1
7,27.740,0
8,29.830,0
9,25.840,0


In [15]:
X["smoker_obese"] = (
    (X["smoker"] == "yes") & (X["bmi"] >= 30)
).astype(int)

In [16]:
X[["bmi", "smoker", "bmi_obese", "smoker_obese"]].head(10)

,bmi,smoker,bmi_obese,smoker_obese
0,27.900,yes,0,0
1,33.770,no,1,0
2,33.000,no,1,0
3,22.705,no,0,0
4,28.880,no,0,0
5,25.740,no,0,0
6,33.440,no,1,0
7,27.740,no,0,0
8,29.830,no,0,0
9,25.840,no,0,0


In [17]:
print(X["bmi_obese"].value_counts())
print()
print(X["smoker_obese"].value_counts())

bmi_obese
1    706
0    631
Name: count, dtype: int64

smoker_obese
0    1192
1     145
Name: count, dtype: int64


In [18]:
X["age_sq"] = X["age"] ** 2

X[["age", "age_sq"]].head()

,age,age_sq
0,19,361
1,18,324
2,28,784
3,33,1089
4,32,1024


In [19]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [56]:
binary_col=["bmi_obese","smoker_obese"]
numerical_col = [
    col for col in X.select_dtypes(include="number").columns
    if col not in binary_col
]

categorical_col=X.select_dtypes(exclude="number").columns.tolist()

In [57]:
numerical_col

['age', 'bmi', 'children', 'age_sq']

In [23]:
categorical_col

['sex', 'smoker', 'region']

In [26]:
X_train, X_test, y_train, y_test=  train_test_split(
          X,
        y,
    test_size=0.2,
    random_state=42,
    stratify=X["smoker"]
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(1069, 9)
(268, 9)
(1069,)
(268,)


In [29]:
print(X_train["smoker"].value_counts(normalize=True)* 100)
print(X_test["smoker"].value_counts(normalize=True)* 100)

smoker
no     79.513564
yes    20.486436
Name: proportion, dtype: float64
smoker
no     79.477612
yes    20.522388
Name: proportion, dtype: float64


In [30]:
X_train.isnull().sum()

age             0
sex             0
bmi             0
children        0
smoker          0
region          0
bmi_obese       0
smoker_obese    0
age_sq          0
dtype: int64

In [33]:
for col in categorical_col:
    print(col, ":", X_train[col].unique())
    print("------------")

sex : <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str
------------
smoker : <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str
------------
region : <ArrowStringArray>
['northwest', 'southeast', 'northeast', 'southwest']
Length: 4, dtype: str
------------


In [31]:
X_test.isnull().sum().sum()

np.int64(0)

In [50]:
from sklearn.preprocessing import  OneHotEncoder,StandardScaler
encoder1=OneHotEncoder(

 drop="first",
    handle_unknown="ignore",
    sparse_output=False
)
X_train_encoded = encoder1.fit_transform(X_train[categorical_col])



In [51]:
X_test_encoded = encoder1.transform(X_test[categorical_col])

In [52]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

(1069, 5)
(268, 5)


In [53]:
encoded_feature_names = encoder1.get_feature_names_out()

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

X_train_encoded_df.head()

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
367,0.0,0.0,1.0,0.0,0.0
418,1.0,0.0,0.0,1.0,0.0
938,1.0,0.0,0.0,1.0,0.0
533,1.0,0.0,0.0,1.0,0.0
354,0.0,0.0,0.0,1.0,0.0


In [66]:
scaler=StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numerical_col])
X_test_scaled = scaler.transform(X_test[numerical_col])

In [67]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=numerical_col,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=numerical_col,
    index=X_test.index
)
X_train_encoded_df.head()

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
367,0.0,0.0,1.0,0.0,0.0
418,1.0,0.0,0.0,1.0,0.0
938,1.0,0.0,0.0,1.0,0.0
533,1.0,0.0,0.0,1.0,0.0
354,0.0,0.0,0.0,1.0,0.0


In [68]:
print("Training Scaled Shape:", X_train_scaled.shape)
print("Testing Scaled Shape:", X_test_scaled.shape)

Training Scaled Shape: (1069, 4)
Testing Scaled Shape: (268, 4)


In [69]:
# Merge Training Data
X_train_final = pd.concat(
    [
        X_train_scaled_df,
        X_train[binary_col],
        X_train_encoded_df
    ],
    axis=1
)

# Merge Testing Data
X_test_final = pd.concat(
    [
        X_test_scaled_df,
        X_test[binary_col],
        X_test_encoded_df
    ],
    axis=1
)

print("Training Final Shape:", X_train_final.shape)
print("Testing Final Shape:", X_test_final.shape)
print(X_train_final.columns.tolist())

Training Final Shape: (1069, 11)
Testing Final Shape: (268, 11)
['age', 'bmi', 'children', 'age_sq', 'bmi_obese', 'smoker_obese', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']


In [70]:
X_train_final.head(10)

,age,bmi,children,age_sq,bmi_obese,smoker_obese,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
367,0.209157,-0.959818,0.769598,0.036308,0,0,0.0,0.0,1.0,0.0,0.0
418,1.775700,1.370265,-0.076801,2.107348,1,0,1.0,0.0,0.0,1.0,0.0
938,0.992428,-0.220931,-0.923201,0.964368,0,0,1.0,0.0,0.0,1.0,0.0
533,-0.146876,0.882057,-0.923201,-0.314490,1,0,1.0,0.0,0.0,1.0,0.0
354,-1.499800,1.225611,-0.923201,-1.242550,1,0,0.0,0.0,0.0,1.0,0.0
1057,-1.072561,1.424511,-0.923201,-1.018750,1,0,0.0,0.0,0.0,1.0,0.0
382,1.134841,0.357686,-0.923201,1.156197,1,0,1.0,0.0,0.0,1.0,0.0
812,-0.787735,-1.365836,0.769598,-0.834026,0,0,1.0,0.0,0.0,0.0,0.0
7,-0.146876,-0.506952,1.615998,-0.314490,0,0,0.0,0.0,1.0,0.0,0.0
216,0.992428,-0.694345,-0.923201,0.964368,0,0,0.0,0.0,1.0,0.0,0.0


In [71]:
print(X_train_final.isnull().sum().sum())
print(X_train_final.index.equals(y_train.index))

0
True


In [74]:
import joblib

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


joblib.dump(X_train_final, PROCESSED_DIR / "X_train_final.pkl")
joblib.dump(X_test_final, PROCESSED_DIR / "X_test_final.pkl")
joblib.dump(y_train, PROCESSED_DIR / "y_train.pkl")
joblib.dump(y_test, PROCESSED_DIR / "y_test.pkl")


joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(encoder1, MODELS_DIR / "encoder.pkl")


joblib.dump(
    {
        "numerical_features": numerical_col,
        "binary_features": binary_col,
        "categorical_features": categorical_col,
        "final_columns": X_train_final.columns.tolist(),
    },
    MODELS_DIR / "feature_config.pkl"
)

print("Saved to:", PROCESSED_DIR)
print("Saved to:", MODELS_DIR)

Saved to: c:\Users\Admin\Desktop\Medical Cost Prediction\data\processed
Saved to: c:\Users\Admin\Desktop\Medical Cost Prediction\models


In [75]:
joblib.dump(X_train, PROCESSED_DIR / "X_train_raw.pkl")
joblib.dump(X_test, PROCESSED_DIR / "X_test_raw.pkl")

['c:\\Users\\Admin\\Desktop\\Medical Cost Prediction\\data\\processed\\X_test_raw.pkl']